# Longitudinal stage IV breast cancer: treatment-driven GRN divergence (Fig. 5)

GRN comparison between the pre-treatment (`pre`) and two post-treatment (`post1`, `post2`) samples of one stage IV breast cancer patient, at the level of the whole tumour (all malignant cells pooled, `T`, versus normal cells, `N`) and of individual clones.

**Inputs** (produced by the standard pipeline, steps 2-4 of the README, run on this dataset as for any other 3CA dataset):
- `ROOT/<CLONE_DIR>/<sample>_clones_with_support.h5ad` for `pre`, `post1`, `post2`: clone-labelled patient files (`obs['clone']` = clone ids and `N`).
- `ROOT/<POOLED_DIR>/<sample>_clones_with_support.h5ad`: the same samples with all malignant cells pooled into one clone `T` (create them with `scripts/02_clone_calling/make_pooled_tumor_h5ad.py`, then run steps 3-4 on that folder).

In [ ]:
import os, itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest, wilcoxon
from sklearn.metrics import r2_score

ROOT = os.environ.get("ITH_STAGE4_ROOT", "stage4_out")
CLONE_DIR = "Breast/Stage4"            # clone-labelled samples
POOLED_DIR = "Breast/Stage4_pooled"    # tumour (T) vs normal (N) samples
TIMEPOINTS = ["pre", "post1", "post2"]
P_THRESH = 0.05
FIG_DIR = "stage4_figs"
os.makedirs(FIG_DIR, exist_ok=True)
sns.set_theme(style="white", font_scale=1.2)

## Load GRNs

In [ ]:
def decode_edgelist(arr, columns, genes, cats):
    df = pd.DataFrame(np.asarray(arr), columns=columns)
    df["clone"] = cats[df["clone"].astype(int)]
    df["source"] = genes[df["source"].astype(int)]
    df["target"] = genes[df["target"].astype(int)]
    return df


def load_sample(directory, sample):
    """{clone: DiGraph} of significant edges (p < P_THRESH; attributes coef_mean, support = bootstrap recoveries out of 100), plus the AnnData."""
    path = os.path.join(ROOT, directory, f"{sample}_clones_with_support.h5ad")
    a = ad.read_h5ad(path)
    edges = a.uns["celloracle_original_filtered"]
    edges = edges[edges["p"] < P_THRESH]
    grns = {}
    for clone, d in edges.groupby("clone"):
        grns[str(clone)] = nx.from_pandas_edgelist(
            d.rename(columns={"bootstrap_support": "support"}), "source", "target",
            edge_attr=["coef_mean", "support"], create_using=nx.DiGraph)
    return grns, a


tumor, clones, adatas = {}, {}, {}
for t in TIMEPOINTS:
    tumor[t], _ = load_sample(POOLED_DIR, t)
    clones[t], adatas[t] = load_sample(CLONE_DIR, t)
    print(t, "tumour edges:", tumor[t]["T"].number_of_edges(), "| clones:", {c: g.number_of_edges() for c, g in clones[t].items()})

## Divergence measures

In [ ]:
def compute_wasserstein(G1, G2, weight, s):
    """L1 distance between the L1-normalised out-edge weight vectors of source s in two GRNs."""
    targets = list({v for _, v in G1.out_edges(s)} | {v for _, v in G2.out_edges(s)})
    v1 = np.array([G1[s][t][weight] if G1.has_edge(s, t) else 0 for t in targets], dtype=float)
    v2 = np.array([G2[s][t][weight] if G2.has_edge(s, t) else 0 for t in targets], dtype=float)
    if v1.sum() > 0:
        v1 = v1 / np.sum(np.abs(v1))
    if v2.sum() > 0:
        v2 = v2 / np.sum(np.abs(v2))
    return np.sum(np.abs(v1 - v2))


def clone_wasserstein(G1, G2, weight="coef_mean"):
    """Mean per-source distance over the sources present in both GRNs."""
    common = {u for u, _ in G1.edges()} & {u for u, _ in G2.edges()}
    if not common:
        return 0
    return sum(compute_wasserstein(G1, G2, weight, s) for s in common) / len(common)


def kth_neighbor_jaccard(G1, G2, node, k=1):
    """Jaccard similarity (plus intersection and union sizes) of the k-hop neighbourhoods of a node in two GRNs."""
    n1 = set(nx.single_source_shortest_path_length(G1, node, cutoff=k)) - {node}
    n2 = set(nx.single_source_shortest_path_length(G2, node, cutoff=k)) - {node}
    union = n1 | n2
    if not union:
        return 0, 0, 0
    inter = n1 & n2
    return len(inter) / len(union), len(inter), len(union)

## Edge overlap between timepoints

In [ ]:
T = {t: tumor[t]["T"] for t in TIMEPOINTS}
for t in TIMEPOINTS:
    print(t, "edges:", T[t].number_of_edges())
print("shared by all three:", nx.intersection_all(list(T.values())).number_of_edges())
for a, b in itertools.combinations(TIMEPOINTS, 2):
    n = nx.intersection(T[a], T[b]).number_of_edges()
    print(f"{a} & {b}: {n}  ({100 * n / T[a].number_of_edges():.2f}% of {a}, {100 * n / T[b].number_of_edges():.2f}% of {b})")

## Tumour-level divergence (Wasserstein) between timepoints, versus clonal divergence

In [ ]:
N = {t: tumor[t]["N"] for t in TIMEPOINTS}
for t in TIMEPOINTS:
    print(f"W({t} tumour, {t} normal) =", round(clone_wasserstein(T[t], N[t]), 4))

W = pd.DataFrame({a: {b: clone_wasserstein(T[a], T[b]) for b in TIMEPOINTS} for a in TIMEPOINTS})
print(W.round(4))
plt.figure(figsize=(4.5, 3.8))
sns.heatmap(W, cmap="RdBu", annot=True, fmt=".2f")
plt.savefig(f"{FIG_DIR}/tumor_wasserstein_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

# clone-vs-clone divergence between timepoints (null for the tumour-level value)
colors = "#5F898D"
for a, b in [("pre", "post1"), ("pre", "post2")]:
    div = [clone_wasserstein(clones[a][c1], clones[b][c2])
           for c1 in clones[a] if c1 != "N" for c2 in clones[b] if c2 != "N"]
    ax = sns.histplot(div, stat="proportion", color=colors, alpha=0.6)
    ax.axvline(W.loc[a, b], linestyle="--", linewidth=2, color=colors)
    ax.tick_params(left=True, bottom=True)
    ax.set_title(f"{a} - {b}: clone pairs (histogram) vs whole tumours (dashed)")
    ax.set_xlabel("Wasserstein distance")
    plt.savefig(f"{FIG_DIR}/clone_pair_divergence_{a}_{b}.png", dpi=300, bbox_inches="tight")
    plt.show()

## k-th order neighbourhood similarity (structural versus functional rewiring)

In [ ]:
KMAX = 7
J = {}
for t2 in ["post1", "post2"]:
    G1, G2 = T["pre"], T[t2]
    common = {u for u, _ in G1.edges()} & {u for u, _ in G2.edges()}
    J[t2] = {v: [kth_neighbor_jaccard(G1, G2, v, k=k)[0] for k in range(1, KMAX + 1)] for v in common}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for k, ax in enumerate(axes.flatten()):
    data = {f"pre vs {t2}": [x[0] - x[k + 1] for x in J[t2].values()] for t2 in ["post1", "post2"]}
    sns.histplot(data, kde=True, ax=ax, stat="density", palette="tab10", legend=(k == 0))
    ax.tick_params(left=True, bottom=True)
    ax.set_title(f"k = {k + 1}")
    ax.set_xlabel("Jaccard(k=1) - Jaccard(k)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/kth_neighbor_jaccard.png", dpi=300)
plt.show()

## Clone size versus divergence from normal, within each timepoint

In [ ]:
D, clone_divergence, binom_result, wilcoxon_result = {}, {}, {}, {}
for t in TIMEPOINTS:
    sizes = adatas[t].obs["clone"].astype(str).value_counts().to_dict()
    sizes = {c: n for c, n in sizes.items() if c in clones[t] and c != "N"}
    D[t], clone_divergence[t] = {}, {}
    for c1, c2 in itertools.combinations(sizes, 2):
        big, small = (c1, c2) if sizes[c1] > sizes[c2] else (c2, c1)
        w_big = clone_wasserstein(clones[t][big], clones[t]["N"])
        w_small = clone_wasserstein(clones[t][small], clones[t]["N"])
        D[t][f"{big}:{small}"] = w_big - w_small
        clone_divergence[t][f"{big}:{small}"] = {"larger": {"size": sizes[big], "W": w_big},
                                                  "smaller": {"size": sizes[small], "W": w_small}}
    vals = list(D[t].values())
    binom_result[t] = binomtest(sum(v > 0 for v in vals), n=len(vals), p=0.5, alternative="greater")
    wilcoxon_result[t] = wilcoxon(vals, alternative="greater")
    print(t, "pairs:", len(vals), "| binomial p =", binom_result[t].pvalue, "| Wilcoxon p =", wilcoxon_result[t].pvalue)

## Centrality: transcription factors whose role changes across timepoints

In [ ]:
CENTRALITY = {"degree": nx.degree_centrality, "betweenness": nx.betweenness_centrality}


def find_pairwise_row_outliers(df, n_std=2):
    """For every pair of rows, fit Y = mX + c across genes and report genes whose residual exceeds n_std standard deviations."""
    out = {}
    for r1, r2 in itertools.combinations(df.index, 2):
        X, Y = df.loc[r1].values, df.loc[r2].values
        m, c = np.polyfit(X, Y, 1)
        resid = Y - (m * X + c)
        sd = np.std(resid)
        out[f"{r1}::{r2}"] = {"R2": r2_score(X, Y), "outliers": [
            {"column": col, "point": (Y[i], X[i]), "residual": resid[i]}
            for i, col in enumerate(df.columns) if abs(resid[i]) > n_std * sd]}
    return out


def centrality_table(graphs, centrality):
    """groups x genes centrality matrix for {group: DiGraph}."""
    cent = {g: CENTRALITY[centrality](G) for g, G in graphs.items()}
    genes = sorted(set().union(*[set(c) for c in cent.values()]))
    return pd.DataFrame([[cent[g].get(n, 0) for n in genes] for g in graphs], index=list(graphs), columns=genes)


# tumour level: pre / post1 / post2
tumor_cent, tumor_outliers = {}, {}
for cent in CENTRALITY:
    tumor_cent[cent] = centrality_table(T, cent).T          # genes x timepoints
    tumor_outliers[cent] = find_pairwise_row_outliers(tumor_cent[cent].T, n_std=2)

# clone level, per timepoint (rows: clones)
clone_cent, clone_outliers = {}, {}
for cent in CENTRALITY:
    clone_cent[cent], clone_outliers[cent] = {}, {}
    for t in TIMEPOINTS:
        clone_cent[cent][t] = centrality_table(clones[t], cent)
        clone_outliers[cent][t] = find_pairwise_row_outliers(clone_cent[cent][t], n_std=3)

In [ ]:
cent = "degree"
for x, y in [("pre", "post1"), ("pre", "post2"), ("post1", "post2")]:
    df = tumor_cent[cent][[x, y]].copy()
    color = np.select([(df[x] == 0) & (df[y] != 0), (df[x] != 0) & (df[y] == 0)], ["#DA2B6B", "#FFC107"], default="#1E88E5")
    fig, ax = plt.subplots()
    ax.scatter(df[x], df[y], c=color)
    lim = float(df.max().max())
    ax.plot([0, lim], [0, lim], color="red")
    ax.tick_params(left=True, bottom=True)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(cent)
    ax.text(0.95 * lim, 0.05 * lim, r"$R^2$=" + f"{r2_score(df[x], df[y]):.4f}", fontsize=14, ha="right")
    plt.savefig(f"{FIG_DIR}/{cent}_{x}_vs_{y}.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(x, y, "outlier genes:", [o["column"] for o in tumor_outliers[cent][f"{x}::{y}"]["outliers"]])

In [ ]:
# genes present at only one timepoint with high degree (e.g. ETS1, KLF6, HEY1 in post1/post2; ZEB1, MEF2A in pre)
deg = tumor_cent["degree"]
raw_deg = pd.DataFrame({t: dict(T[t].degree()) for t in TIMEPOINTS}).fillna(0).astype(int)
absent_pre = raw_deg[(raw_deg["pre"] == 0) & ((raw_deg["post1"] > 0) | (raw_deg["post2"] > 0))]
print("top degree among genes absent from pre:")
print(absent_pre.sort_values(["post1", "post2"], ascending=False).head(10))
only_pre = raw_deg[(raw_deg["pre"] > 0) & (raw_deg["post1"] == 0) & (raw_deg["post2"] == 0)]
print("top degree among genes only in pre:")
print(only_pre.sort_values("pre", ascending=False).head(10))


def mean_support(gene):
    """Mean bootstrap support (out of 100) over a gene's edges at each timepoint."""
    out = {}
    for t in TIMEPOINTS:
        G = T[t]
        s = [d["support"] for u, v, d in G.edges(data=True) if gene in (u, v)]
        out[t] = round(float(np.mean(s)), 1) if s else 0.0
    return out


for g in ["ETS1", "KLF6", "HEY1", "ZEB1", "MEF2A"]:
    print(g, mean_support(g))

In [ ]:
# clone-specific betweenness of selected transcription factors, normalised by each clone's maximum
GENES = ["FOXF2", "GLI2", "CREM", "EGR1"]      # edit to genes of interest
for a, b in [("pre", "post1"), ("pre", "post2"), ("post1", "post2")]:
    rows = []
    for t in (a, b):
        table = clone_cent["betweenness"][t]
        table = table.loc[[c for c in table.index if c != "N"]]
        norm = table.div(table.max(axis=1).replace(0, np.nan), axis=0)
        for clone in norm.index:
            for g in GENES:
                rows.append({"clone": clone, "gene": g, "treatment": t,
                             "betweenness": norm.loc[clone, g] if g in norm.columns else -0.1})
    ax = sns.catplot(data=pd.DataFrame(rows), x="betweenness", y="clone", hue="gene", col="treatment",
                     marker="*", s=100, palette="tab10")
    ax.refline(x=0, color="red")
    ax.savefig(f"{FIG_DIR}/clone_betweenness_{a}_{b}.png", dpi=300)
    plt.show()